# E se 5% da carteira fosse Bitcoin?

**Python Finance Lab · Aula 8** · *Finanças na prática, notebook por notebook.*

---

## Parte 1 · A pergunta

Na aula 7, vimos que o Bitcoin oscila muito mais que qualquer ação da carteira. A conclusão apressada seria: então ele só aumenta o risco. Mas a aula 5 ensinou que o risco de uma carteira depende também de como os ativos andam juntos. Um ativo muito volátil, mas que se move de um jeito diferente do resto, pode mudar o conjunto menos do que parece.

Nesta aula, vamos fazer o teste com uma fatia pequena. Partimos dos mesmos R\\$ 2.000 da aula 1 e tiramos uma parte para o Bitcoin: 1%, 5% ou 10%. O resto continua na carteira de 10 ações. Para cada fatia, comparamos duas regras: deixar a fatia correr livremente ou trazê-la de volta ao alvo uma vez por ano, como o rebalanceamento da aula 4.

Antes das contas, há dois problemas práticos para resolver: o Bitcoin precisa estar em reais, e os calendários não combinam. A bolsa fecha nos fins de semana e nos feriados, e o mercado de cripto nunca fecha. Juntar os dois sem criar dias que não existiram é a novidade de Python desta aula.

*Conteúdo exclusivamente educacional. Não constitui recomendação de investimento.*

## Parte 2 · Conceitos

### Bloco 2 · O vocabulário da aula

**Bitcoin em reais.** Há dois caminhos. O direto é o par **BTCBRL**, negociado em reais na própria Binance. O indireto é multiplicar o preço em USDT pelo dólar: se um bitcoin vale 60 mil USDT e o dólar está em R\\$ 5,50, ele vale cerca de R\\$ 330 mil. Os dois caminhos dão números próximos, e conferir um com o outro é a validação da aula.

**Alocação pequena.** Uma fatia de poucos por cento num ativo muito arriscado. A ideia é que, mesmo se o ativo perder muito, o estrago no conjunto seja limitado: 5% que caem pela metade custam 2,5% da carteira.

**Calendários diferentes.** As ações têm preço só nos dias de pregão da B3, cerca de 250 por ano. O Bitcoin tem preço todos os dias, 365 por ano. Para comparar os dois, usamos só os dias em que os dois têm preço: os dias de pregão.

**Dias falsos.** Um erro comum é juntar as duas séries por todas as datas e preencher os buracos das ações nos fins de semana com o último preço. Isso cria dias em que a ação "não oscilou", que nunca aconteceram, e faz o risco das ações parecer menor do que é.

**Fuso horário.** O candle diário da Binance fecha à meia-noite UTC, que são 21h em Brasília. Então, o candle que a Binance chama de "1º de outubro" fecha às 21h do dia 1º, no horário de Brasília. Ele é o fechamento do Bitcoin naquele dia, algumas horas depois do fechamento da bolsa.

**Rebalancear uma fatia.** Se o Bitcoin dispara, a fatia de 5% pode virar 15% sem que ninguém faça nada: é a deriva dos pesos da aula 4. Rebalancear devolve a fatia ao alvo, vendendo parte do que subiu. Sem rebalancear, a alocação "pequena" pode deixar de ser pequena.

## Parte 3 · Fórmulas

### Bloco 3 · As contas da aula

**Bitcoin em reais, pelo caminho indireto**, supondo 1 USDT ≈ 1 dólar:

$$P^{\text{BRL}}_t \approx P^{\text{USDT}}_t \times \text{dólar}_t$$

**Valor da carteira** com uma parte em ações e outra em Bitcoin, e o peso do Bitcoin:

$$V_t = V^{\text{ações}}_t + V^{\text{BTC}}_t \qquad\qquad x_t = \frac{V^{\text{BTC}}_t}{V_t}$$

A cada dia, cada parte cresce com o próprio retorno: $V^{\text{ações}}_t = V^{\text{ações}}_{t-1}(1 + r^{\text{ações}}_t)$, e o mesmo para o Bitcoin. No rebalanceamento, as partes voltam ao alvo $x$: $V^{\text{BTC}}_t = x \cdot V_t$ e $V^{\text{ações}}_t = (1 - x) \cdot V_t$.

*Exemplo à mão:* R\\$ 2.000 com 5% em Bitcoin são R\\$ 1.900 em ações e R\\$ 100 em Bitcoin. Se as ações sobem 10% e o Bitcoin dobra, a carteira vai para R\\$ 2.090 + R\\$ 200 = R\\$ 2.290. O peso do Bitcoin passou a $200 / 2.290 = 8{,}7\%$.

- **Rebalanceando:** o Bitcoin volta a 5% de R\\$ 2.290, ou seja, R\\$ 114,50, e as ações ficam com R\\$ 2.175,50.
- **Sem rebalancear:** se o Bitcoin dobrar de novo, ele sozinho vai a R\\$ 400, e o peso dele sobe para mais de 16%.

## Parte 4 · Implementação

### Bloco 4 · Importar as bibliotecas

As bibliotecas de sempre. Nesta aula, não há gráfico de candles, então o plotly não é necessário.

> **O que esperar:** nada. Se aparecer `ModuleNotFoundError`, instale com `%pip install yfinance pandas matplotlib requests` e reinicie o kernel (menu Kernel → Restart Kernel).

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import requests

### Bloco 5 · As premissas

As ações, as datas e o capital da aula 1, as fatias de Bitcoin que vamos testar, numa lista, e o endereço da API da Binance da aula 7.

> **O que esperar:** nada.

In [ ]:
ACOES = [
    "ITUB4.SA", "VALE3.SA", "PETR4.SA", "WEGE3.SA", "ABEV3.SA",
    "EGIE3.SA", "SUZB3.SA", "RADL3.SA", "VIVT3.SA", "TOTS3.SA",
]
DATA_INICIO = "2021-10-01"
DATA_FIM = "2026-09-30"
CAPITAL = 2000
FATIAS_BTC = [0, 0.01, 0.05, 0.10]
BASE_URL = "https://data-api.binance.vision"

### Bloco 6 · Ações, CDI e a carteira da aula 1

Os preços das 10 ações e a carteira da aula 1, os R\\$ 2.000 divididos igualmente e sem rebalancear. Nesta aula, ela funciona como um bloco só: a "parte em ações".

Vamos buscar duas séries no Banco Central, o CDI e o dólar. Por isso, o código da aula 1 vira uma função, `baixar_sgs`, que recebe o número da série e as datas. A série 12 é o CDI; a série 1, o dólar comercial de venda, em reais.

> **O que esperar:** o número de dias da carteira e do CDI, cerca de 1.250 em cada. Repare que a URL da função usa duas f-strings: o número da série também entra no endereço.

In [ ]:
# Ações e a carteira da aula 1
dados = yf.download(ACOES, start=DATA_INICIO, auto_adjust=True, progress=False)
precos = dados["Close"].loc[DATA_INICIO:DATA_FIM].ffill()
precos.columns = precos.columns.str.replace(".SA", "")
carteira = (precos / precos.iloc[0]).mean(axis=1) * CAPITAL


def baixar_sgs(serie, inicio, fim):
    """Baixa uma série do SGS do Banco Central e devolve uma Series com datas e números."""
    inicio_bc = pd.to_datetime(inicio).strftime("%d/%m/%Y")
    fim_bc = pd.to_datetime(fim).strftime("%d/%m/%Y")
    url = (
        f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie}/dados"
        f"?formato=json&dataInicial={inicio_bc}&dataFinal={fim_bc}"
    )
    resposta = requests.get(url, timeout=60)
    resposta.raise_for_status()
    tabela = pd.DataFrame(resposta.json())
    tabela["data"] = pd.to_datetime(tabela["data"], format="%d/%m/%Y")
    tabela["valor"] = tabela["valor"].astype(float)
    return tabela.set_index("data")["valor"]


cdi_diario = baixar_sgs(12, DATA_INICIO, DATA_FIM)
cota_cdi = (1 + cdi_diario / 100).cumprod()
print("Dias da carteira:", len(carteira), "| dias do CDI:", len(cdi_diario))

### Bloco 7 · O Bitcoin em reais, da Binance

Versões simplificadas das duas funções da aula 7, `organizar_candles` e `baixar_candles`: a paginação é a mesma, mas sem as validações extras daquela aula (`validar_candles`). Depois, o download do par BTCBRL. Na aula 10, funções reaproveitadas assim vão para um arquivo próprio; por enquanto, copiamos.

> **O que esperar:** duas linhas de progresso, com 1.826 candles no total, e os três últimos candles do BTCBRL, com preços em reais.

In [ ]:
COLUNAS = [
    "abertura_ms", "abertura", "maxima", "minima", "fechamento", "volume",
    "fechamento_ms", "volume_usdt", "negocios", "compra_base", "compra_usdt", "ignorar",
]


def organizar_candles(linhas):
    """Transforma a lista de listas da Binance numa tabela de candles com datas e números."""
    tabela = pd.DataFrame(linhas, columns=COLUNAS)
    tabela["data"] = pd.to_datetime(tabela["abertura_ms"], unit="ms", utc=True)
    colunas_preco = ["abertura", "maxima", "minima", "fechamento", "volume"]
    tabela[colunas_preco] = tabela[colunas_preco].astype(float)
    return tabela.set_index("data")[colunas_preco]


def baixar_candles(par, inicio, fim, intervalo="1d"):
    """Baixa todos os candles de um par entre duas datas, em páginas de até 1.000."""
    inicio_ms = int(pd.Timestamp(inicio, tz="UTC").timestamp() * 1000)
    fim_ms = int((pd.Timestamp(fim, tz="UTC") + pd.Timedelta(days=1)).timestamp() * 1000) - 1
    linhas = []
    while inicio_ms <= fim_ms:
        parametros = {
            "symbol": par, "interval": intervalo,
            "startTime": inicio_ms, "endTime": fim_ms, "limit": 1000,
        }
        resposta = requests.get(f"{BASE_URL}/api/v3/klines", params=parametros, timeout=30)
        resposta.raise_for_status()
        pagina = resposta.json()
        if len(pagina) == 0:
            break
        linhas = linhas + pagina
        inicio_ms = pagina[-1][0] + 1
        print(f"{par}: {len(pagina)} candles nesta página, {len(linhas)} no total")
    return organizar_candles(linhas)


btc_brl = baixar_candles("BTCBRL", DATA_INICIO, DATA_FIM)
btc_brl.tail(3)

**Depois de rodar — resultado da execução registrada no vídeo.** Vieram **1.826 candles**, em duas páginas (1.000 + 826). O candle de 30/09/2026 fechou em **R\\$ 434.201,00 por bitcoin** no par BTCBRL.

### Bloco 8 · Do horário da Binance para o dia de Brasília

O índice dos candles é o horário de **abertura**, em UTC. Para comparar com a bolsa, queremos o dia do **fechamento**, no horário de Brasília. São três passos:

1. **Abertura mais um dia é o fechamento.** `btc_brl.index + pd.Timedelta(days=1)` leva cada candle ao seu horário de fechamento: meia-noite UTC do dia seguinte.
2. **De UTC para Brasília.** `tz_convert("America/Sao_Paulo")` converte o horário para o fuso de Brasília, o mesmo de São Paulo. A meia-noite UTC vira 21h do dia anterior: exatamente o dia do candle.
3. **Só a data.** `tz_localize(None)` tira a etiqueta de fuso, e `normalize()` zera a hora, deixando só o dia. Agora as datas têm o mesmo formato das datas das ações.

A tabelinha mostra os três horários dos primeiros candles, lado a lado, para conferir a conversão.

> **O que esperar:** a tabela de conversão, em que o candle aberto em 2021-10-01 00:00 UTC fecha em 2021-10-02 00:00 UTC, ou seja, em 2021-10-01 21:00 no horário de Brasília (`-03:00`). Depois, os três primeiros fechamentos do Bitcoin, com datas simples: 2021-10-01, 2021-10-02 e 2021-10-03.

In [ ]:
fechamento_utc = btc_brl.index + pd.Timedelta(days=1)
fechamento_brasilia = fechamento_utc.tz_convert("America/Sao_Paulo")

conversao = pd.DataFrame({
    "abertura (UTC)": btc_brl.index[:3],
    "fechamento (UTC)": fechamento_utc[:3],
    "fechamento (Brasília)": fechamento_brasilia[:3],
})
print(conversao)

btc = btc_brl["fechamento"].copy()
btc.index = fechamento_brasilia.tz_localize(None).normalize()
btc.index.name = "data"
btc.head(3)

### Bloco 9 · O calendário da bolsa manda

Agora, os dois calendários. Primeiro, o jeito errado: juntar a carteira e o Bitcoin por todas as datas com `pd.concat`, como nas aulas anteriores, mas sem o `join="inner"`. O `sort=True` só mantém as datas em ordem. Aparecem os fins de semana e feriados, com a carteira vazia nesses dias. Preencher esses buracos com o último preço inventaria dias em que as ações não oscilaram.

O jeito certo é o **`reindex`**: ele reorganiza uma série para um índice escolhido por nós. `btc.reindex(carteira.index)` fica só com os fechamentos do Bitcoin nos dias de pregão da B3, na ordem deles. O que o Bitcoin fez no fim de semana não se perde: ele aparece inteiro no retorno de segunda-feira, que vai do fechamento de sexta ao de segunda.

Se algum dia de pregão não tiver preço de Bitcoin, o `reindex` deixa um `NaN`, e a contagem mostra isso. Como o cripto não fecha, esperamos zero.

> **O que esperar:** cerca de 1.826 linhas no jeito errado, com quase 580 dias de carteira vazia, e, no jeito certo, o mesmo número de dias de pregão da carteira, sem nenhum dia sem preço de Bitcoin.

In [ ]:
juntas_errado = pd.concat([carteira, btc], axis=1, sort=True)
print("Linhas juntando todos os dias:", len(juntas_errado))
print("Dias com a carteira vazia:", juntas_errado.iloc[:, 0].isna().sum())
print()

btc_b3 = btc.reindex(carteira.index)
print("Dias de pregão:", len(btc_b3))
print("Dias de pregão sem preço de BTC:", btc_b3.isna().sum())

### Bloco 10 · Bitcoin × carteira

Antes de misturar, comparamos os dois lado a lado: quanto R\\$ 2.000 viraram em cada um, e a correlação entre os retornos diários, como na aula 5. A correlação vai dizer se o Bitcoin anda junto com as ações ou de outro jeito.

> **O que esperar:** as duas curvas, com o Bitcoin oscilando muito mais, e a correlação entre os retornos diários.

In [ ]:
lado_a_lado = pd.DataFrame({
    "Carteira": carteira,
    "Bitcoin": btc_b3 / btc_b3.iloc[0] * CAPITAL,
})

ax = lado_a_lado.plot(
    figsize=(10, 5),
    style=["-", "--"],
    color=["#2a78d6", "#eda100"],
    linewidth=2,
    title="R$ 2.000 na carteira de ações e em Bitcoin, em reais",
)
ax.axhline(CAPITAL, color="gray", linewidth=0.8)
ax.set_ylabel("Patrimônio (R$)")
ax.set_xlabel("")
ax.grid(alpha=0.3)
plt.show()

retornos = lado_a_lado.pct_change().dropna()
print("Correlação diária entre o Bitcoin e a carteira:", round(retornos.corr().iloc[0, 1], 2))

**Depois de rodar — resultado da execução registrada no vídeo.** A correlação diária entre o Bitcoin e a carteira foi de **0,04**. Na aula 5, a correlação média entre os pares de ações foi de **0,20**. São comparações diferentes: aqui o Bitcoin é comparado com a carteira inteira. Correlação baixa indica pouca associação linear neste recorte, não independência nem proteção garantida.

### Bloco 11 · A função que mistura ações e Bitcoin

A simulação é diária e segue o bloco 3. A cada dia de pregão, cada parte cresce com o próprio retorno. Se a regra for rebalancear, a fatia volta ao alvo no primeiro pregão de cada ano novo.

- `pct_change().fillna(0)` calcula os retornos diários e troca o `NaN` do primeiro dia por zero: no primeiro dia, ninguém ganhou nem perdeu nada.
- A decisão de rebalancear usa o operador **`and`** ("e"): só rebalanceia se a regra pedir **e** o ano tiver mudado desde o pregão anterior. É o `or` do exercício 1 da aula 4, só que exigindo as duas condições ao mesmo tempo.
- `ano_anterior` guarda o ano do pregão anterior, para a comparação do dia seguinte.

A função devolve um dicionário, como a da aula 4: o patrimônio e o peso do Bitcoin em cada dia.

> **O que esperar:** nada. A função foi ensinada ao Python.

In [ ]:
def simular_com_btc(acoes, btc, fatia, rebalancear, capital=CAPITAL):
    """Patrimônio diário com uma fatia em Bitcoin e o resto na carteira de ações."""
    ret_acoes = acoes.pct_change().fillna(0)
    ret_btc = btc.pct_change().fillna(0)
    valor_acoes = capital * (1 - fatia)
    valor_btc = capital * fatia
    patrimonio = []
    peso_btc = []
    ano_anterior = acoes.index[0].year

    for data in acoes.index:
        valor_acoes = valor_acoes * (1 + ret_acoes.loc[data])
        valor_btc = valor_btc * (1 + ret_btc.loc[data])
        total = valor_acoes + valor_btc
        if rebalancear and data.year != ano_anterior:
            valor_acoes = total * (1 - fatia)
            valor_btc = total * fatia
        ano_anterior = data.year
        patrimonio.append(total)
        peso_btc.append(valor_btc / total)

    return {
        "patrimonio": pd.Series(patrimonio, index=acoes.index),
        "peso_btc": pd.Series(peso_btc, index=acoes.index),
    }

### Bloco 12 · Testando a função com o exemplo à mão

O teste repete o exemplo do bloco 3, com quatro pregões. Do primeiro para o segundo, as ações sobem 10% e o Bitcoin dobra. O terceiro pregão é o primeiro de um ano novo, sem variação de preço; e, no quarto, o Bitcoin dobra de novo.

- **Rebalanceando:** no terceiro pregão, o Bitcoin volta a R\\$ 114,50; no quarto, dobra para R\\$ 229. A carteira termina em R\\$ 2.175,50 + R\\$ 229 = R\\$ 2.404,50.
- **Sem rebalancear:** o Bitcoin vai de R\\$ 200 a R\\$ 400, e a carteira termina em R\\$ 2.090 + R\\$ 400 = R\\$ 2.490.

> **O que esperar:** `[2000.0, 2290.0, 2290.0, 2404.5]` e `[2000.0, 2290.0, 2290.0, 2490.0]`.

In [ ]:
datas_teste = pd.to_datetime(["2021-12-30", "2021-12-31", "2022-01-03", "2022-01-04"])
acoes_teste = pd.Series([100.0, 110.0, 110.0, 110.0], index=datas_teste)
btc_teste = pd.Series([100.0, 200.0, 200.0, 400.0], index=datas_teste)

com = simular_com_btc(acoes_teste, btc_teste, 0.05, True, capital=2000)
sem = simular_com_btc(acoes_teste, btc_teste, 0.05, False, capital=2000)
print(com["patrimonio"].round(2).tolist())   # esperado: [2000, 2290, 2290, 2404.5]
print(sem["patrimonio"].round(2).tolist())   # esperado: [2000, 2290, 2290, 2490]

### Bloco 13 · As oito carteiras

Quatro fatias de Bitcoin, cada uma com as duas regras: são oito simulações, guardadas num dicionário. O nome de cada uma é montado com uma f-string: `{fatia:.0%}` mostra 0,05 como 5%, e o `if` escolhe o complemento do nome.

> **O que esperar:** oito linhas, com o patrimônio final de cada combinação. As duas de 0% são iguais: sem Bitcoin, não há o que rebalancear.

In [ ]:
simulacoes = {}
for fatia in FATIAS_BTC:
    for rebalancear in [False, True]:
        if rebalancear:
            regra = "rebalanceando"
        else:
            regra = "sem rebalancear"
        nome = f"{fatia:.0%} BTC, {regra}"
        simulacoes[nome] = simular_com_btc(carteira, btc_b3, fatia, rebalancear)

for nome, r in simulacoes.items():
    print(f"{nome}: R$ {r['patrimonio'].iloc[-1]:.2f}")

**Depois de rodar — resultado da execução registrada no vídeo.** Patrimônio final de cada combinação:

| Fatia de BTC | Sem rebalancear | Rebalanceando |
|---|---|---|
| 0% | R\\$ 3.712,68 | R\\$ 3.712,68 |
| 1% | R\\$ 3.709,20 | R\\$ 3.764,46 |
| 5% | R\\$ 3.695,30 | R\\$ 3.962,78 |
| 10% | R\\$ 3.677,92 | R\\$ 4.189,71 |

### Bloco 14 · A tabela de risco e retorno

A tabela que a aula promete: as medidas da aula 2 para cada combinação. Os dados são diários, nos dias de pregão, então a volatilidade usa o padrão de 252. A régua do Sharpe é a cota do CDI. Atenção: esta coluna é uma razão didática, a diferença entre os retornos anuais compostos da carteira e do CDI dividida pela volatilidade. O Sharpe clássico usa os excessos de retorno de cada período.

> **O que esperar:** uma tabela com oito linhas e seis colunas.

In [ ]:
def retorno_anual(serie):
    """Retorno médio por ano, com juros compostos (anos contados em dias corridos)."""
    anos = (serie.index[-1] - serie.index[0]).days / 365.25
    return (serie.iloc[-1] / serie.iloc[0]) ** (1 / anos) - 1


def volatilidade_anual(serie, periodos_por_ano=252):
    """Desvio padrão dos retornos, levado para a escala de um ano."""
    return serie.pct_change().dropna().std() * periodos_por_ano ** 0.5


def queda_maxima(serie):
    """A pior queda do período, do pico ao fundo."""
    return (serie / serie.cummax() - 1).min()


def sharpe(serie, cdi, periodos_por_ano=252):
    """Retorno acima do CDI por ano, para cada ponto de volatilidade."""
    return (retorno_anual(serie) - retorno_anual(cdi)) / volatilidade_anual(serie, periodos_por_ano)


tabela = pd.DataFrame(index=list(simulacoes))
for nome, r in simulacoes.items():
    p = r["patrimonio"]
    tabela.loc[nome, "Patrimônio final (R$)"] = p.iloc[-1]
    tabela.loc[nome, "Retorno ao ano (%)"] = retorno_anual(p) * 100
    tabela.loc[nome, "Volatilidade ao ano (%)"] = volatilidade_anual(p) * 100
    tabela.loc[nome, "Queda máxima (%)"] = queda_maxima(p) * 100
    tabela.loc[nome, "Sharpe"] = sharpe(p, cota_cdi)
    tabela.loc[nome, "Peso final do BTC (%)"] = r["peso_btc"].iloc[-1] * 100

tabela.round(2)

**Depois de rodar — resultado da execução registrada no vídeo.** Rebalanceando, a volatilidade foi de **14,27%** (0%) para **14,16%** (1%), **13,94%** (5%) e **14,16%** (10%): não cresceu na proporção da fatia. A queda máxima piorou a cada fatia: **−16,13%**, **−16,38%**, **−17,35%** e **−18,60%**. O Sharpe didático foi de **0,04** para **0,06**, **0,15** e **0,23**. Sem rebalancear, as fatias de 1%, 5% e 10% terminaram abaixo da carteira sem Bitcoin, com Sharpe de 0,04, 0,03 e 0,02.

### Bloco 15 · As curvas, com rebalanceamento anual

O gráfico mostra as quatro fatias com a regra de rebalanceamento. A carteira sem Bitcoin mantém o azul de sempre. Cada fatia de Bitcoin ganha uma cor própria e um estilo de linha diferente, para não se confundirem quando as curvas se cruzam: amarelo para 1%, verde para 5% e vermelho para 10%, a fatia mais arriscada.

> **O que esperar:** quatro curvas parecidas, que se afastam mais nos períodos de grandes movimentos do Bitcoin.

In [ ]:
curvas = pd.DataFrame()
for fatia in FATIAS_BTC:
    curvas[f"{fatia:.0%} BTC"] = simulacoes[f"{fatia:.0%} BTC, rebalanceando"]["patrimonio"]

ax = curvas.plot(
    figsize=(10, 5),
    style=["-", ":", "--", "-."],
    color=["#2a78d6", "#eda100", "#008300", "#e34948"],
    linewidth=2,
    title="R$ 2.000 com fatias de Bitcoin, rebalanceando todo ano",
)
ax.axhline(CAPITAL, color="gray", linewidth=0.8)
ax.set_ylabel("Patrimônio (R$)")
ax.set_xlabel("")
ax.grid(alpha=0.3)
plt.show()

**Depois de rodar — resultado da execução registrada no vídeo.** Até 2023, as quatro curvas quase se sobrepõem. A partir de 2024, com as altas do Bitcoin, as curvas com mais Bitcoin se afastam para cima; em 2022, nas quedas, ficaram um pouco abaixo. Observe o caminho, não só o último ponto: terminar acima não comprova menor risco.

### Bloco 16 · Quanto a fatia de 5% cresceu?

Por fim, a deriva. O gráfico mostra o peso do Bitcoin na carteira de 5%, dia a dia, com e sem rebalanceamento, e a linha cinza do alvo de 5%. Sem rebalancear, a alocação "pequena" fica à mercê do Bitcoin.

> **O que esperar:** duas linhas que saem juntas de 5%. A linha verde e tracejada, a mesma cor dos 5% no gráfico anterior, volta a 5% a cada virada de ano, formando "dentes de serra"; a roxa e contínua vai para onde o Bitcoin a levar.

In [ ]:
pesos = pd.DataFrame({
    "sem rebalancear": simulacoes["5% BTC, sem rebalancear"]["peso_btc"] * 100,
    "rebalanceando": simulacoes["5% BTC, rebalanceando"]["peso_btc"] * 100,
})

ax = pesos.plot(
    figsize=(10, 4),
    style=["-", "--"],
    color=["#4a3aa7", "#008300"],
    linewidth=2,
    title="Peso do Bitcoin na carteira de 5%",
)
ax.axhline(5, color="gray", linewidth=0.8)
ax.set_ylabel("Peso do BTC (%)")
ax.set_xlabel("")
ax.grid(alpha=0.3)
plt.show()

print("Maior peso sem rebalancear:", round(pesos["sem rebalancear"].max(), 1), "%")
print("Maior peso rebalanceando:  ", round(pesos["rebalanceando"].max(), 1), "%")

**Depois de rodar — resultado da execução registrada no vídeo.** Sem rebalancear, o peso do Bitcoin variou de **1,53%** a **8,69%** (o código imprime só o máximo, arredondado para 8,7%; o mínimo aparece no gráfico). Rebalanceando uma vez por ano, o maior peso chegou a **12,99%**: depois de cada ajuste, novas altas podem ampliar a fatia até o ajuste seguinte. O rebalanceamento anual não mantém 5% o tempo todo nem garante o menor pico de exposição.

## Parte 5 · Validação

### Bloco 17 · Com 0% de Bitcoin, a carteira da aula 1?

Sem Bitcoin, a função tem de devolver exatamente a carteira da aula 1, com e sem a regra de rebalanceamento. A maior diferença tem de ser zero, ou um número minúsculo.

> **O que esperar:** `R$ 0.0` nas duas linhas.

In [ ]:
for regra in ["sem rebalancear", "rebalanceando"]:
    diferenca = (simulacoes[f"0% BTC, {regra}"]["patrimonio"] - carteira).abs().max()
    print(f"0% BTC, {regra}: maior diferença para a aula 1 = R$ {round(diferenca, 6)}")

### Bloco 18 · O BTCBRL confere com BTCUSDT × dólar?

A conferência com outra fonte: calculamos o Bitcoin em reais pelo caminho indireto do bloco 3, multiplicando o BTCUSDT da Binance pelo dólar do Banco Central, e comparamos com o BTCBRL.

- O BTCUSDT passa pela mesma conversão de datas do bloco 8.
- O dólar só existe nos dias úteis, então o `reindex` leva o BTCUSDT para as datas do dólar.
- A diferença de cada dia, em %, é o preço indireto dividido pelo direto, menos 1. Resumimos com a mediana das diferenças absolutas, que não se deixa levar por poucos dias extremos, e com a maior.

Diferenças pequenas são esperadas, por três motivos: o dólar da série 1 é apurado no meio do dia, e o Bitcoin, às 21h; o USDT não vale exatamente um dólar; e o par em reais tem menos negócios, e seu preço pode se afastar um pouco do internacional.

> **O que esperar:** o número de dias úteis comparados, uma diferença mediana pequena e a maior diferença do período.

In [ ]:
btc_usdt = baixar_candles("BTCUSDT", DATA_INICIO, DATA_FIM)["fechamento"]
btc_usdt.index = (btc_usdt.index + pd.Timedelta(days=1)).tz_convert("America/Sao_Paulo").tz_localize(None).normalize()

dolar = baixar_sgs(1, DATA_INICIO, DATA_FIM)
indireto = btc_usdt.reindex(dolar.index) * dolar
direto = btc.reindex(dolar.index)

diferenca_pct = (indireto / direto - 1) * 100
print("Dias comparados:", diferenca_pct.notna().sum())
print(f"Diferença mediana: {diferenca_pct.abs().median():.2f}%")
print(f"Maior diferença:   {diferenca_pct.abs().max():.2f}%")

**Depois de rodar — resultado da execução registrada no vídeo.** Foram **1.255 dias úteis** comparados. A diferença absoluta mediana foi de **0,49%**, e a maior, de **3,68%**. Isso mostra coerência entre os dois caminhos, mas não demonstra arbitragem nem valida o preço em outra corretora.

## Parte 6 · Interpretação

### Bloco 19 · O que os números dizem

A correlação diária entre o Bitcoin e a carteira foi de **0,04**. Com 5% de Bitcoin e rebalanceamento anual, os R\\$ 2.000 viraram **R\\$ 3.962,78**, contra **R\\$ 3.712,68** da carteira sem Bitcoin. A volatilidade passou de **14,27%** para **13,94%** ao ano, mas a queda máxima foi de **−16,13%** para **−17,35%**. Sem rebalancear, a mesma fatia terminou em **R\\$ 3.695,30**, e o peso do Bitcoin variou de **1,53%** a **8,69%**; com o ajuste anual, chegou a **12,99%**.

Perguntas que orientam a interpretação:

1. **A fatia pequena mudou muito o risco?** Compare a volatilidade com 0%, 1%, 5% e 10%. Ela cresceu na mesma proporção da fatia, mais, ou menos? A correlação explica a resposta.
2. **E o retorno?** O resultado depende muito do período. Comece a simulação num pico ou num fundo do Bitcoin e a conclusão pode se inverter (exercício 3).
3. **Rebalancear fez diferença?** Compare as duas regras na mesma fatia. Rebalancear vende Bitcoin depois das altas e compra depois das quedas, e devolve a fatia ao tamanho planejado nas datas de ajuste. Entre um ajuste e outro, a fatia fica livre: neste período, o pico de peso foi maior com o ajuste anual do que sem ele.
4. **O pior caso é aceitável?** Se o Bitcoin pesar 5% naquele momento e o resto não mudar, uma queda de 80% dele custa cerca de 4% da carteira. É essa a lógica de uma alocação pequena. Mas o peso muda com os preços (aqui chegou a quase 13%), então a conta é uma referência, não um limite para a perda total.

## Parte 7 · Limitações

### Bloco 20 · O que esta análise não mostra

- **Um único período.** Cinco anos de um ativo que já teve altas e quedas de mais de 70%. Outra data de início muda muito o resultado.
- **Horários diferentes.** O Bitcoin é medido às 21h, e as ações, no fechamento da bolsa, à tarde. As notícias da noite entram no Bitcoin de um dia e nas ações do dia seguinte. Isso pode alterar a correlação medida, mas não dá para afirmar que sempre a reduz.
- **Preço de uma corretora.** O par BTCBRL tem menos negócios que o BTCUSDT, e o preço de uma única corretora pode se afastar do de outras.
- **Custos, impostos e custódia.** Comprar, guardar e vender criptomoedas tem custos e regras de imposto próprias, que não entraram na conta.
- **A parte em ações não é rebalanceada por dentro.** As 10 ações seguem a carteira da aula 1, sem rebalanceamento entre elas; só a divisão entre ações e Bitcoin é rebalanceada.
- **Os números do vídeo.** Na execução gravada, os preços da aula 1 e as respostas das APIs foram reaproveitados de arquivos salvos, para manter a comparação. Rodando hoje, as fontes podem devolver números um pouco diferentes.
- **Risco além da volatilidade.** O cripto tem riscos que as medidas da aula 2 não capturam: falhas de corretoras, perda de chaves, mudanças de regulação.

## Parte 8 · Exercícios

### Bloco 21 · Para praticar

1. **Ethereum no lugar do Bitcoin.** Baixe o par `"ETHBRL"` e repita a aula. O ETH diversificou mais ou menos que o BTC?
2. **Rebalanceamento mensal.** Mude a condição para rebalancear quando o mês mudar: `data.month != mes_anterior`, com uma variável `mes_anterior`. Compare com o anual.
3. **Outra data de início.** Comece a simulação em 2022-11-01, perto de um fundo do Bitcoin, em vez de 2021-10-01. As conclusões se invertem?
4. **Uma fatia de 2%.** Acrescente 0,02 à lista `FATIAS_BTC`. Onde ela fica na tabela, entre 1% e 5%?
5. **Rebalancear por faixa.** Rebalanceie só quando o peso do Bitcoin sair da faixa entre metade e o dobro do alvo, como no exercício 1 da aula 4.

## Resumo

### Bloco 22 · O que aprendemos

**Em finanças:**

- O Bitcoin em reais pode vir do par BTCBRL ou de BTCUSDT multiplicado pelo dólar; os dois caminhos dão valores próximos.
- O efeito de um ativo no risco da carteira depende do tamanho da fatia, da volatilidade dele e da correlação com o resto.
- O peso de uma fatia muda com os preços. O rebalanceamento anual devolve a fatia ao alvo só nas datas de ajuste; entre elas, ela pode crescer.
- Juntar séries com calendários diferentes exige escolher um calendário e não inventar dias.

**Em Python:**

- `tz_convert` muda o fuso de datas com fuso; `tz_localize(None)` tira o fuso; `normalize` zera a hora.
- `reindex` reorganiza uma série para um índice escolhido, deixando `NaN` onde não há valor.
- `fillna(0)` troca valores vazios por zero.
- O operador `and` exige duas condições verdadeiras ao mesmo tempo; o `or`, pelo menos uma.

## Gancho para a aula 9

### Bloco 23 · Próxima aula

Até aqui, todas as carteiras ficaram sempre investidas, atravessando cada queda até o fim. A tentação natural é perguntar: e se desse para sair antes das quedas e voltar antes das altas?

Uma das regras mais populares para isso usa a média móvel: ficar investido quando o preço está acima da média dos últimos 200 dias e ir para a renda fixa quando está abaixo. Na próxima aula, vamos testar essa regra de forma honesta, com custos, sem olhar o futuro e sem escolher a "melhor" regra depois de saber o resultado.

## Glossário da aula

| Termo | Significado |
|---|---|
| BTCBRL | Par de negociação do Bitcoin cotado em reais |
| Alocação pequena | Fatia de poucos por cento num ativo muito arriscado |
| Calendário de negociação | Os dias em que um ativo tem preço; na B3, só os dias de pregão |
| Dia falso | Data inventada ao preencher buracos de uma série com o último valor |
| Fuso horário | Região com a mesma hora oficial; Brasília é UTC−3 |
| `tz_convert` | Converte datas de um fuso horário para outro |
| `reindex` | Reorganiza uma série para um índice escolhido |
| `fillna` | Preenche valores vazios com um valor escolhido |
| `and` | Operador que exige que duas condições sejam verdadeiras |

---

*Python Finance Lab · Conteúdo exclusivamente educacional. Não constitui recomendação de investimento.*